# **PUC-Rio | Departamento de Engenharia Industrial**
# **ENG 4560: Projeto Integrado VI - Distribuição Física**

---

# **Aula 8 - Heurísticas de melhoria (busca local)**  
**Prof. Marcello Congro (marcellocongro@puc-rio.br)**

---

### Objetivo da aula

Na aula anterior, construímos soluções iniciais para o problema de roteirização da Prolog usando heurísticas construtivas, como:

- Nearest Neighbor;
- Clarke-Wright Savings.

Ao final daquela aula, cada equipe salvou a solução construída em um arquivo `.json`, por exemplo:

- `solution_nn_het.json`;
- `solution_cw_het.json`.

Nesta aula, vamos carregar uma dessas soluções e utilizá-la como ponto de partida para aplicar heurísticas de melhoria, também chamadas de métodos de busca local.

A pergunta central desta aula é:

> Uma solução construída rapidamente pode ser melhorada com movimentos simples?

### O que faremos

1. Carregar a mesma instância usada na aula anterior;
2. Carregar a solução construtiva salva em JSON;
3. Padronizar a solução para busca local;
4. Calcular métricas da solução inicial;
5. Aplicar 2-opt dentro das rotas;
6. Aplicar relocate entre rotas;
7. Comparar a solução inicial com as soluções melhoradas;
8. Discutir se a melhoria obtida justifica o esforço computacional adicional.

### Observação importante

A busca local não substitui a heurística construtiva. Ela depende dela.

Por isso, a qualidade da solução final pode depender bastante da solução inicial escolhida.

In [ ]:
# ============================================================
# (1) IMPORTAÇÕES E PREPARAÇÃO DO AMBIENTE
# ============================================================
# Nesta aula, vamos trabalhar com busca local aplicada a soluções
# previamente construídas.
#
# A lógica principal será:
# 1) carregar a instância;
# 2) carregar a solução construtiva salva em JSON;
# 3) aplicar movimentos de melhoria;
# 4) comparar antes e depois.

import numpy as np
import pandas as pd
import json
import os
import time
import copy
import matplotlib.pyplot as plt

from google.colab import files

In [ ]:
# ============================================================
# (2) UPLOAD DA INSTÂNCIA E DA SOLUÇÃO CONSTRUTIVA
# ============================================================
# Faça upload dos arquivos da instância:
#
# - nodes.csv
# - D.npy
# - Cvar.npy
# - q.npy
# - s.npy
# - params.json
#
# Além disso, faça upload de UMA solução construtiva gerada na aula anterior:
#
# - solution_nn_het.json
# ou
# - solution_cw_het.json
#
# A solução JSON deve corresponder à mesma instância carregada.

!rm -f *.csv *.npy *.json

print("Faça upload da instância e do arquivo JSON da solução construtiva.")
uploaded = files.upload()

print("\nArquivos disponíveis no ambiente:")
print(os.listdir())

In [ ]:
# ============================================================
# (3) LEITURA E VALIDAÇÃO DA INSTÂNCIA
# ============================================================
# Assim como nas aulas anteriores, precisamos garantir que a instância
# foi carregada corretamente antes de aplicar qualquer procedimento.

nodes = pd.read_csv("nodes.csv")
D = np.load("D.npy")          # matriz de distâncias (km)
Cvar = np.load("Cvar.npy")    # matriz de custo variável
q = np.load("q.npy")          # demandas (kg)
s = np.load("s.npy")          # tempos de serviço (h)

params = json.loads(open("params.json", "r", encoding="utf-8").read())

n = len(nodes)

assert D.shape == (n, n), "Dimensão incorreta da matriz D"
assert Cvar.shape == (n, n), "Dimensão incorreta da matriz Cvar"
assert q.shape == (n,), "Dimensão incorreta do vetor q"
assert s.shape == (n,), "Dimensão incorreta do vetor s"
assert nodes.loc[0, "id"] == 0, "O depósito deve ter id = 0"

print(f"Instância carregada com sucesso: {n-1} clientes + depósito")

In [ ]:
# ============================================================
# (4) PARÂMETROS LOGÍSTICOS
# ============================================================
# Mantemos os mesmos parâmetros do problema da Prolog usados nas aulas
# anteriores. Eles serão necessários para verificar capacidade, jornada
# e custo das rotas após os movimentos de busca local.

Q = {
    "FIO": 650.0,
    "VUC": 3000.0
}

f = {
    "FIO": 250.0,
    "VUC": 550.0
}

cost_per_km = 1.50
v_kmh = 40.0
H = 8.0

print("Capacidades:", Q)
print("Custos fixos:", f)
print("Custo variável por km:", cost_per_km)
print("Velocidade média (km/h):", v_kmh)
print("Jornada máxima (h):", H)

In [ ]:
# ============================================================
# (5) CARREGAMENTO DA SOLUÇÃO CONSTRUTIVA
# ============================================================
# Nesta célula, identificamos o arquivo de solução salvo na aula anterior.
#
# Ele deve ter nome parecido com:
# - solution_nn_het.json
# - solution_cw_het.json
#
# Como também existe params.json, não podemos simplesmente carregar
# qualquer arquivo .json. Vamos procurar arquivos que começam com "solution_".

solution_files = [
    file for file in os.listdir()
    if file.startswith("solution_") and file.endswith(".json")
]

if len(solution_files) == 0:
    raise FileNotFoundError(
        "Nenhum arquivo de solução encontrado. "
        "Faça upload de solution_nn_het.json ou solution_cw_het.json."
    )

if len(solution_files) > 1:
    print("Mais de uma solução encontrada:")
    for idx, file in enumerate(solution_files):
        print(idx, "-", file)

    selected_idx = 0
    print("\nUsando automaticamente o primeiro arquivo da lista.")
    print("Altere selected_idx manualmente se quiser usar outro arquivo.")
    solution_filename = solution_files[selected_idx]

else:
    solution_filename = solution_files[0]

with open(solution_filename, "r", encoding="utf-8") as f_json:
    solution_loaded = json.load(f_json)

print("Solução carregada:", solution_filename)
print("Número de rotas carregadas:", len(solution_loaded))

In [ ]:
# ============================================================
# (6) IDENTIFICAÇÃO DO MÉTODO CONSTRUTIVO DE ORIGEM
# ============================================================
# Apenas para fins de relatório e comparação, vamos identificar se a
# solução veio do NN ou do Clarke-Wright pelo nome do arquivo.

if "nn" in solution_filename.lower():
    INITIAL_METHOD = "Nearest Neighbor"
elif "cw" in solution_filename.lower() or "clarke" in solution_filename.lower():
    INITIAL_METHOD = "Clarke-Wright Savings"
else:
    INITIAL_METHOD = "Método construtivo não identificado"

print("Método construtivo de origem:", INITIAL_METHOD)

In [ ]:
# ============================================================
# (7) PADRONIZAÇÃO DA SOLUÇÃO PARA BUSCA LOCAL
# ============================================================
# Os notebooks de NN e Clarke-Wright salvam a solução como uma lista
# de dicionários.
#
# Para esta aula, vamos manter também uma lista de dicionários, mas
# garantindo que cada rota tenha pelo menos:
#
# - route_id
# - vehicle
# - route
#
# Isso facilita a aplicação dos movimentos 2-opt e relocate.

def standardize_solution(solution):
    standardized = []

    for idx, item in enumerate(solution, start=1):
        route_id = item.get("route_id", idx)
        vehicle = item["vehicle"]
        route = item["route"]

        standardized.append({
            "route_id": route_id,
            "vehicle": vehicle,
            "route": route
        })

    return standardized


solution_initial = standardize_solution(solution_loaded)

print("Solução inicial padronizada:")

for item in solution_initial:
    print(
        f"Rota {item['route_id']} | "
        f"Veículo: {item['vehicle']} | "
        f"Sequência: {item['route']}"
    )

In [ ]:
# ============================================================
# (8) FUNÇÕES AUXILIARES DE ROTA
# ============================================================
# Estas funções serão usadas para calcular distância, carga, tempo,
# custo e viabilidade antes e depois dos movimentos de busca local.

def route_distance(route, D):
    dist = 0.0
    for a in range(len(route) - 1):
        i = route[a]
        j = route[a + 1]
        dist += D[i, j]
    return dist


def route_load(route, q):
    clients = [node for node in route if node != 0]
    return sum(q[i] for i in clients)


def route_service_time(route, s):
    clients = [node for node in route if node != 0]
    return sum(s[i] for i in clients)


def route_total_time(route, D, s, v_kmh):
    dist = route_distance(route, D)
    t_mov = dist / v_kmh
    t_serv = route_service_time(route, s)
    t_total = t_mov + t_serv
    return t_total, t_mov, t_serv


def route_cost(route, vehicle_type, D, cost_per_km, fixed_costs):
    dist = route_distance(route, D)
    return fixed_costs[vehicle_type] + dist * cost_per_km


def is_route_feasible(route, vehicle_type, D, q, s, Q, H, v_kmh):
    load = route_load(route, q)
    total_time, _, _ = route_total_time(route, D, s, v_kmh)

    feasible_capacity = load <= Q[vehicle_type] + 1e-6
    feasible_time = total_time <= H + 1e-6

    return feasible_capacity and feasible_time

In [ ]:
# ============================================================
# (9) MÉTRICAS DA SOLUÇÃO
# ============================================================

def solution_metrics(solution, D, q, s, cost_per_km, fixed_costs, Q, H, v_kmh):
    total_dist = 0.0
    total_cost = 0.0
    total_time = 0.0

    violations_capacity = 0
    violations_time = 0

    vehicle_count = {"FIO": 0, "VUC": 0}

    for item in solution:
        route = item["route"]
        vehicle = item["vehicle"]

        dist = route_distance(route, D)
        cost = route_cost(route, vehicle, D, cost_per_km, fixed_costs)
        total_time_route, _, _ = route_total_time(route, D, s, v_kmh)

        total_dist += dist
        total_cost += cost
        total_time += total_time_route

        if vehicle in vehicle_count:
            vehicle_count[vehicle] += 1
        else:
            vehicle_count[vehicle] = 1

        if route_load(route, q) > Q[vehicle] + 1e-6:
            violations_capacity += 1

        if total_time_route > H + 1e-6:
            violations_time += 1

    return {
        "n_routes": len(solution),
        "n_fio": vehicle_count.get("FIO", 0),
        "n_vuc": vehicle_count.get("VUC", 0),
        "total_distance_km": total_dist,
        "total_cost_rs": total_cost,
        "total_time_h": total_time,
        "capacity_violations": violations_capacity,
        "time_violations": violations_time
    }


metrics_initial = solution_metrics(
    solution=solution_initial,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    Q=Q,
    H=H,
    v_kmh=v_kmh
)

print("Métricas da solução inicial:")
print(metrics_initial)

In [ ]:
# ============================================================
# (10) TABELA RESUMO DA SOLUÇÃO INICIAL
# ============================================================

def build_solution_summary(solution, D, q, s, cost_per_km, fixed_costs, H, v_kmh):
    rows = []

    for idx, item in enumerate(solution, start=1):
        route = item["route"]
        vehicle = item["vehicle"]

        t_total, t_mov, t_serv = route_total_time(route, D, s, v_kmh)

        rows.append({
            "route_id": item.get("route_id", idx),
            "vehicle": vehicle,
            "route": str(route),
            "n_clients": len([node for node in route if node != 0]),
            "load_kg": round(route_load(route, q), 2),
            "distance_km": round(route_distance(route, D), 2),
            "total_time_h": round(t_total, 2),
            "travel_time_h": round(t_mov, 2),
            "service_time_h": round(t_serv, 2),
            "total_cost_rs": round(route_cost(route, vehicle, D, cost_per_km, fixed_costs), 2),
            "violates_H": t_total > H + 1e-6
        })

    return pd.DataFrame(rows)


df_initial = build_solution_summary(
    solution=solution_initial,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    H=H,
    v_kmh=v_kmh
)

df_initial

In [ ]:
# ============================================================
# (11) VISUALIZAÇÃO DA SOLUÇÃO INICIAL
# ============================================================

def plot_solution(solution, nodes, title):
    x_coords = nodes["lon"].values
    y_coords = nodes["lat"].values

    plt.figure(figsize=(9, 9))

    plt.scatter(x_coords[1:], y_coords[1:], color="blue", label="Clientes")
    plt.scatter(x_coords[0], y_coords[0], color="red", s=150, label="Depósito")

    first_fio = True
    first_vuc = True

    for item in solution:
        route = item["route"]
        vehicle = item["vehicle"]

        if vehicle == "FIO":
            linestyle = "--"
            label = "FIO" if first_fio else None
            first_fio = False
        else:
            linestyle = "-"
            label = "VUC" if first_vuc else None
            first_vuc = False

        for a in range(len(route) - 1):
            i = route[a]
            j = route[a + 1]

            plt.plot(
                [x_coords[i], x_coords[j]],
                [y_coords[i], y_coords[j]],
                linestyle=linestyle,
                label=label if a == 0 else None
            )

    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.grid(True)
    plt.legend()
    plt.show()


plot_solution(
    solution_initial,
    nodes,
    f"Solução inicial — {INITIAL_METHOD}"
)

# (12) BUSCA LOCAL 1 — 2-opt

O movimento 2-opt atua dentro de uma única rota.

A ideia é remover dois arcos e reconectar a rota invertendo um trecho intermediário.

Esse movimento é especialmente útil quando a rota apresenta cruzamentos ou uma sequência de atendimento pouco eficiente.

Nesta implementação, vamos aplicar 2-opt em cada rota separadamente, mantendo:

- o mesmo conjunto de clientes em cada rota;
- o mesmo veículo alocado;
- a mesma capacidade;
- a mesma jornada máxima.

Ou seja, o 2-opt melhora a ordem interna da rota, mas não troca clientes entre veículos.

In [ ]:
# ============================================================
# (13) MOVIMENTO 2-OPT
# ============================================================
# O 2-opt inverte um trecho da rota.
#
# Exemplo:
# rota original: [0, A, B, C, D, 0]
# se invertermos o trecho entre B e D:
# nova rota:     [0, A, D, C, B, 0]

def two_opt_swap(route, i, k):
    # TODO (ALUNO):
    # implementar a inversão do segmento entre i e k.
    #
    # GABARITO:
    new_route = route[:i] + route[i:k+1][::-1] + route[k+1:]

    return new_route

In [ ]:
# ============================================================
# (14) 2-OPT EM UMA ROTA
# ============================================================

def improve_route_2opt(route, vehicle_type, D, q, s, Q, H, v_kmh):
    best_route = route.copy()
    best_distance = route_distance(best_route, D)

    improved = True

    while improved:
        improved = False

        # O depósito inicial e final não são alterados.
        for i in range(1, len(best_route) - 2):
            for k in range(i + 1, len(best_route) - 1):

                candidate = two_opt_swap(best_route, i, k)

                if not is_route_feasible(candidate, vehicle_type, D, q, s, Q, H, v_kmh):
                    continue

                candidate_distance = route_distance(candidate, D)

                if candidate_distance < best_distance - 1e-6:
                    best_route = candidate
                    best_distance = candidate_distance
                    improved = True
                    break

            if improved:
                break

    return best_route

In [ ]:
# ============================================================
# (15) APLICAÇÃO DO 2-OPT NA SOLUÇÃO INTEIRA
# ============================================================

def apply_2opt_to_solution(solution, D, q, s, Q, H, v_kmh):
    improved_solution = []

    for item in solution:
        vehicle = item["vehicle"]
        route = item["route"]

        improved_route = improve_route_2opt(
            route=route,
            vehicle_type=vehicle,
            D=D,
            q=q,
            s=s,
            Q=Q,
            H=H,
            v_kmh=v_kmh
        )

        improved_solution.append({
            "route_id": item["route_id"],
            "vehicle": vehicle,
            "route": improved_route
        })

    return improved_solution


start_2opt = time.time()

solution_2opt = apply_2opt_to_solution(
    solution=solution_initial,
    D=D,
    q=q,
    s=s,
    Q=Q,
    H=H,
    v_kmh=v_kmh
)

elapsed_2opt = time.time() - start_2opt

metrics_2opt = solution_metrics(
    solution=solution_2opt,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    Q=Q,
    H=H,
    v_kmh=v_kmh
)

print("Métricas após 2-opt:")
print(metrics_2opt)
print("Tempo computacional 2-opt (s):", elapsed_2opt)

In [ ]:
# ============================================================
# (16) TABELA RESUMO APÓS 2-OPT
# ============================================================

df_2opt = build_solution_summary(
    solution=solution_2opt,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    H=H,
    v_kmh=v_kmh
)

df_2opt

# (17) BUSCA LOCAL 2 — Relocate

O movimento relocate atua entre rotas.

A ideia é retirar um cliente de uma rota de origem e tentar inseri-lo em outra rota, em alguma posição viável.

Esse movimento é importante porque, diferentemente do 2-opt, ele altera a composição das rotas.

Enquanto o 2-opt melhora a ordem interna de uma rota, o relocate pode redistribuir clientes entre veículos.

Nesta implementação, vamos testar movimentos de relocate entre rotas diferentes, sempre respeitando:

- capacidade do veículo de destino;
- jornada máxima da rota de destino;
- jornada máxima da rota de origem após a remoção.

In [ ]:
# ============================================================
# (18) CUSTO TOTAL DA SOLUÇÃO
# ============================================================

def total_solution_cost(solution, D, cost_per_km, fixed_costs):
    total = 0.0

    for item in solution:
        route = item["route"]
        vehicle = item["vehicle"]
        total += route_cost(route, vehicle, D, cost_per_km, fixed_costs)

    return total

In [ ]:
# ============================================================
# (19) MOVIMENTO RELOCATE ENTRE ROTAS
# ============================================================

def relocate_customer(route_from, route_to, idx_from, idx_to):
    """
    Move um cliente da rota de origem para a rota de destino.

    idx_from: posição do cliente removido da rota de origem.
    idx_to: posição onde o cliente será inserido na rota de destino.
    """

    new_from = route_from.copy()
    new_to = route_to.copy()

    customer = new_from.pop(idx_from)
    new_to.insert(idx_to, customer)

    return new_from, new_to

In [ ]:
# ============================================================
# (20) BUSCA LOCAL COM RELOCATE ENTRE ROTAS
# ============================================================
# Estratégia:
#
# - percorremos pares de rotas;
# - removemos um cliente da rota de origem;
# - testamos sua inserção em diferentes posições da rota de destino;
# - aceitamos a primeira melhoria encontrada;
# - repetimos até não haver mais melhora.
#
# Esta é uma estratégia de "first improvement".

def improve_solution_relocate(solution, D, q, s, Q, f, cost_per_km, H, v_kmh):
    best_solution = copy.deepcopy(solution)
    best_cost = total_solution_cost(best_solution, D, cost_per_km, f)

    improved = True

    while improved:
        improved = False

        for a in range(len(best_solution)):
            route_from = best_solution[a]["route"]
            vehicle_from = best_solution[a]["vehicle"]

            for b in range(len(best_solution)):
                if a == b:
                    continue

                route_to = best_solution[b]["route"]
                vehicle_to = best_solution[b]["vehicle"]

                # Não removemos o depósito inicial nem final.
                for idx_from in range(1, len(route_from) - 1):

                    # Inserção pode ocorrer antes do depósito final da rota de destino.
                    for idx_to in range(1, len(route_to)):

                        candidate_solution = copy.deepcopy(best_solution)

                        cand_from = candidate_solution[a]["route"]
                        cand_to = candidate_solution[b]["route"]

                        new_from, new_to = relocate_customer(
                            route_from=cand_from,
                            route_to=cand_to,
                            idx_from=idx_from,
                            idx_to=idx_to
                        )

                        candidate_solution[a]["route"] = new_from
                        candidate_solution[b]["route"] = new_to

                        # Remover rota vazia do tipo [0, 0].
                        candidate_solution = [
                            item for item in candidate_solution
                            if len([node for node in item["route"] if node != 0]) > 0
                        ]

                        # Verificar viabilidade de todas as rotas.
                        feasible = True

                        for item in candidate_solution:
                            if not is_route_feasible(
                                route=item["route"],
                                vehicle_type=item["vehicle"],
                                D=D,
                                q=q,
                                s=s,
                                Q=Q,
                                H=H,
                                v_kmh=v_kmh
                            ):
                                feasible = False
                                break

                        if not feasible:
                            continue

                        candidate_cost = total_solution_cost(
                            candidate_solution,
                            D,
                            cost_per_km,
                            f
                        )

                        if candidate_cost < best_cost - 1e-6:
                            best_solution = copy.deepcopy(candidate_solution)
                            best_cost = candidate_cost
                            improved = True
                            break

                    if improved:
                        break

                if improved:
                    break

            if improved:
                break

    # Renumerar rotas
    for rid, item in enumerate(best_solution, start=1):
        item["route_id"] = rid

    return best_solution

In [ ]:
# ============================================================
# (21) APLICAÇÃO DO RELOCATE APÓS 2-OPT
# ============================================================

start_relocate = time.time()

solution_2opt_relocate = improve_solution_relocate(
    solution=solution_2opt,
    D=D,
    q=q,
    s=s,
    Q=Q,
    f=f,
    cost_per_km=cost_per_km,
    H=H,
    v_kmh=v_kmh
)

elapsed_relocate = time.time() - start_relocate

metrics_2opt_relocate = solution_metrics(
    solution=solution_2opt_relocate,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    Q=Q,
    H=H,
    v_kmh=v_kmh
)

print("Métricas após 2-opt + relocate:")
print(metrics_2opt_relocate)
print("Tempo computacional relocate (s):", elapsed_relocate)

In [ ]:
# ============================================================
# (22) TABELA RESUMO APÓS 2-OPT + RELOCATE
# ============================================================

df_2opt_relocate = build_solution_summary(
    solution=solution_2opt_relocate,
    D=D,
    q=q,
    s=s,
    cost_per_km=cost_per_km,
    fixed_costs=f,
    H=H,
    v_kmh=v_kmh
)

df_2opt_relocate

In [ ]:
# ============================================================
# (23) COMPARAÇÃO CONSOLIDADA
# ============================================================

comparison = pd.DataFrame([
    {
        "Solução": f"Inicial ({INITIAL_METHOD})",
        "Nº rotas": metrics_initial["n_routes"],
        "FIO": metrics_initial["n_fio"],
        "VUC": metrics_initial["n_vuc"],
        "Distância (km)": metrics_initial["total_distance_km"],
        "Custo (R$)": metrics_initial["total_cost_rs"],
        "Tempo total rotas (h)": metrics_initial["total_time_h"],
        "Viol. capacidade": metrics_initial["capacity_violations"],
        "Viol. jornada": metrics_initial["time_violations"],
        "Tempo comp. (s)": 0.0
    },
    {
        "Solução": "Após 2-opt",
        "Nº rotas": metrics_2opt["n_routes"],
        "FIO": metrics_2opt["n_fio"],
        "VUC": metrics_2opt["n_vuc"],
        "Distância (km)": metrics_2opt["total_distance_km"],
        "Custo (R$)": metrics_2opt["total_cost_rs"],
        "Tempo total rotas (h)": metrics_2opt["total_time_h"],
        "Viol. capacidade": metrics_2opt["capacity_violations"],
        "Viol. jornada": metrics_2opt["time_violations"],
        "Tempo comp. (s)": elapsed_2opt
    },
    {
        "Solução": "Após 2-opt + relocate",
        "Nº rotas": metrics_2opt_relocate["n_routes"],
        "FIO": metrics_2opt_relocate["n_fio"],
        "VUC": metrics_2opt_relocate["n_vuc"],
        "Distância (km)": metrics_2opt_relocate["total_distance_km"],
        "Custo (R$)": metrics_2opt_relocate["total_cost_rs"],
        "Tempo total rotas (h)": metrics_2opt_relocate["total_time_h"],
        "Viol. capacidade": metrics_2opt_relocate["capacity_violations"],
        "Viol. jornada": metrics_2opt_relocate["time_violations"],
        "Tempo comp. (s)": elapsed_relocate
    }
])

comparison

In [ ]:
# ============================================================
# (24) GANHOS PERCENTUAIS
# ============================================================

cost_initial = metrics_initial["total_cost_rs"]
cost_2opt = metrics_2opt["total_cost_rs"]
cost_final = metrics_2opt_relocate["total_cost_rs"]

dist_initial = metrics_initial["total_distance_km"]
dist_2opt = metrics_2opt["total_distance_km"]
dist_final = metrics_2opt_relocate["total_distance_km"]

gain_cost_2opt = 100 * (cost_initial - cost_2opt) / cost_initial
gain_cost_final = 100 * (cost_initial - cost_final) / cost_initial

gain_dist_2opt = 100 * (dist_initial - dist_2opt) / dist_initial
gain_dist_final = 100 * (dist_initial - dist_final) / dist_initial

print(f"Ganho de custo após 2-opt: {gain_cost_2opt:.2f}%")
print(f"Ganho de custo após 2-opt + relocate: {gain_cost_final:.2f}%")
print()
print(f"Ganho de distância após 2-opt: {gain_dist_2opt:.2f}%")
print(f"Ganho de distância após 2-opt + relocate: {gain_dist_final:.2f}%")

In [ ]:
# ============================================================
# (25) VISUALIZAÇÃO DA SOLUÇÃO FINAL
# ============================================================

plot_solution(
    solution_2opt_relocate,
    nodes,
    f"Solução final — {INITIAL_METHOD} + 2-opt + relocate"
)

In [ ]:
# ============================================================
# (26) SALVAR SOLUÇÃO FINAL
# ============================================================
# A solução final pode ser salva para compor o relatório da Sprint 2.

output_filename = "solution_busca_local_final.json"

with open(output_filename, "w", encoding="utf-8") as f_out:
    json.dump(solution_2opt_relocate, f_out, indent=4, ensure_ascii=False)

files.download(output_filename)

print(f"Solução final salva em: {output_filename}")

# (27) DISCUSSÃO FINAL

Responda no relatório da Sprint 2:

1. Qual solução construtiva foi usada como ponto de partida?
   - Nearest Neighbor?
   - Clarke-Wright Savings?

2. A aplicação do 2-opt melhorou a solução?
   - Em custo?
   - Em distância?
   - Em estrutura visual das rotas?

3. O relocate trouxe ganho adicional após o 2-opt?

4. A busca local alterou:
   - o número de rotas?
   - a composição da frota?
   - a distância total?
   - o custo total?

5. O ganho obtido justifica o tempo computacional adicional?

6. Se a equipe testar as duas soluções iniciais, responda:
   - NN + busca local ou CW + busca local gerou melhor resultado?
   - Qual solução inicial apresentou maior potencial de melhoria?
   - Qual abordagem você recomendaria à Prolog?

### Reflexão final

A busca local não deve ser avaliada apenas pelo ganho percentual.

Ela deve ser analisada como parte de um processo de decisão:

heurística construtiva → solução inicial → melhoria local → recomendação operacional.